In [1]:
import os
import glob
import numpy as np
import pandas as pd

# PCA matrices — same ones used during generation
_BASE = "../file_processing/pca_csv"
eigenvectors = pd.read_csv(f"{_BASE}/pc_vals.csv", index_col=0).values  # (6, 500)
means        = pd.read_csv(f"{_BASE}/pc_val_means.csv")["pc_mean"].values  # (500,)

PC_COLS  = [f"PC{i}" for i in range(1, 7)]
N_POINTS = 250

RESULTS_DIR = "pca_llm_results"
OUT_DIR     = "pca_llm_results/processed"
os.makedirs(OUT_DIR, exist_ok=True)

In [2]:
# Load every CSV.gz in the results directory and tag with source filename
files = sorted(glob.glob(f"{RESULTS_DIR}/*.csv.gz"))
print(f"Found {len(files)} file(s):")
for f in files:
    print(f"  {f}")

raw = pd.concat(
    [pd.read_csv(f).assign(source_file=os.path.basename(f)) for f in files],
    ignore_index=True,
)
print(f"\nTotal vase rows loaded: {len(raw):,}")
raw.head(2)

Found 4 file(s):
  pca_llm_results/gpt_results.csv.gz
  pca_llm_results/gpt_results_v1.csv.gz
  pca_llm_results/gpt_results_v2.csv.gz
  pca_llm_results/gpt_results_v3.csv.gz

Total vase rows loaded: 33,330


,phase,session_id,session_started_at,model,round,gen,display_position,is_selected,is_parent,PC1,...,PC6,rt_tokens,budget_tokens,budget_utilisation,wall_time_s,input_tokens,output_tokens,stop_reason,timestamp,source_file
0,main,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,2026-05-27T00:06:21.836637,gpt-5.4-nano-2026-03-17,1,1.0,1,False,False,-3.374880,...,0.896722,69,5000,0.0164,2.169575,1558,82,completed,2026-05-27T00:06:24.054391,gpt_results.csv.gz
1,main,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,2026-05-27T00:06:21.836637,gpt-5.4-nano-2026-03-17,1,1.0,2,True,False,-7.832178,...,2.368173,69,5000,0.0164,2.169575,1558,82,completed,2026-05-27T00:06:24.054391,gpt_results.csv.gz


In [3]:
# Vectorised XY reconstruction — no Python loop over rows
pc_arr = raw[PC_COLS].values                        # (n_vases, 6)
shapes = pc_arr @ eigenvectors + means              # (n_vases, 500)
x_mat  = shapes[:, :N_POINTS]                       # (n_vases, 250)
y_mat  = shapes[:, N_POINTS:2 * N_POINTS]           # (n_vases, 250)

n = len(raw)
rep_idx = np.repeat(np.arange(n), N_POINTS)

xy = raw.iloc[rep_idx].reset_index(drop=True)
xy["x"]           = x_mat.ravel()
xy["y"]           = y_mat.ravel()
xy["point_index"] = np.tile(np.arange(1, N_POINTS + 1), n)

# --- Align column names and dtypes to match human xy_data.parquet ---

# generation_key: "round_0_gen_1" (0-indexed rounds) for main; "tournament" for final phase
is_main = xy["phase"] == "main"
xy["generation_key"] = "tournament"
xy.loc[is_main, "generation_key"] = (
    "round_" + (xy.loc[is_main, "round"].astype(int) - 1).astype(str)
    + "_gen_" + xy.loc[is_main, "gen"].astype(int).astype(str)
)

# vase: "vase_1", "vase_2" etc. (from display_position)
xy["vase"] = "vase_" + xy["display_position"].astype(int).astype(str)

# rename to match human schema
xy = xy.rename(columns={
    "is_selected": "is_selected_vase",
    "is_parent":   "is_parent_vase",
    "round":       "play_counter",
    "session_id":  "participant_id",
})

# participant_id = session_id for AI (each session is one synthetic participant)
xy["session_id"] = xy["participant_id"]

# is_parent_vase: tournament rows carry None — fill to False so dtype is bool
xy["is_parent_vase"] = xy["is_parent_vase"].fillna(False).astype(bool)

# session_started_at: human parquet stores this as float64 Unix timestamp — match that
xy["session_started_at"] = (
    pd.to_datetime(xy["session_started_at"], utc=True).astype("int64") / 1e9
)

KEEP_COLS = [
    "generation_key", "vase", "x", "y", "point_index",
    "is_selected_vase", "is_parent_vase",
    "participant_id", "session_id", "play_counter",
    "phase", "model", "gen",
    "wall_time_s", "rt_tokens", "budget_tokens", "budget_utilisation",
    "input_tokens", "output_tokens", "stop_reason",
    "session_started_at", "timestamp", "source_file",
]
xy = xy[KEEP_COLS]

print(f"XY rows: {len(xy):,}")
print(f"session_started_at dtype: {xy['session_started_at'].dtype}  sample: {xy['session_started_at'].iloc[0]:.3f}")
print(f"is_parent_vase dtype: {xy['is_parent_vase'].dtype}")
xy.head(3)

XY rows: 8,332,500
session_started_at dtype: float64  sample: 1779840.382
is_parent_vase dtype: bool


,generation_key,vase,x,y,point_index,is_selected_vase,is_parent_vase,participant_id,session_id,play_counter,...,wall_time_s,rt_tokens,budget_tokens,budget_utilisation,input_tokens,output_tokens,stop_reason,session_started_at,timestamp,source_file
0,round_0_gen_1,vase_1,-0.000167,1.605543,1,False,False,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,1,...,2.169575,69,5000,0.0164,1558,82,completed,1.779840e+06,2026-05-27T00:06:24.054391,gpt_results.csv.gz
1,round_0_gen_1,vase_1,0.049104,1.605331,2,False,False,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,1,...,2.169575,69,5000,0.0164,1558,82,completed,1.779840e+06,2026-05-27T00:06:24.054391,gpt_results.csv.gz
2,round_0_gen_1,vase_1,0.098312,1.605205,3,False,False,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,1,...,2.169575,69,5000,0.0164,1558,82,completed,1.779840e+06,2026-05-27T00:06:24.054391,gpt_results.csv.gz


In [4]:
out_path = f"{OUT_DIR}/ai_xy_data.parquet"
xy.to_parquet(out_path, index=False)
print(f"Saved {len(xy):,} rows → {out_path}")

# Quick sanity check
check = pd.read_parquet(out_path)
print(f"\nShape:   {check.shape}")
print(f"Models:  {check['model'].unique().tolist()}")
print(f"Phases:  {check['phase'].unique().tolist()}")
print(f"Participants: {check['participant_id'].nunique()}")
check.groupby(["model", "phase"])[["is_selected_vase"]].count().rename(columns={"is_selected_vase": "rows"})

Saved 8,332,500 rows → pca_llm_results/processed/ai_xy_data.parquet

Shape:   (8332500, 23)
Models:  ['gpt-5.4-nano-2026-03-17']
Phases:  ['main', 'tournament']
Participants: 606


rows
model                   phase              
gpt-5.4-nano-2026-03-17 main        7575000
                        tournament   757500

In [5]:
# ── Build AI PC parquet matching human pc_data.parquet structure ──────────────
# Human format: long — one row per PC per vase (6 rows per vase).
# AI raw format: wide — PC1…PC6 as separate columns.
# Steps: rename shared cols → build generation_key → melt PCs → fix dtypes.

pc = raw.copy()

# generation_key: 0-indexed rounds for main, "tournament" for final phase
is_main_pc = pc["phase"] == "main"
pc["generation_key"] = "tournament"
pc.loc[is_main_pc, "generation_key"] = (
    "round_" + (pc.loc[is_main_pc, "round"].astype(int) - 1).astype(str)
    + "_gen_" + pc.loc[is_main_pc, "gen"].astype(int).astype(str)
)

# vase column
pc["vase"] = "vase_" + pc["display_position"].astype(int).astype(str)

# Rename shared columns to match human schema
pc = pc.rename(columns={
    "is_selected": "is_selected_vase",
    "is_parent":   "is_parent_vase",
    "round":       "play_counter",
    "session_id":  "participant_id",
})
pc["session_id"] = pc["participant_id"]

# is_parent_vase: tournament rows carry None — fill to False so dtype is bool
pc["is_parent_vase"] = pc["is_parent_vase"].fillna(False).astype(bool)

# session_started_at → float64 Unix timestamp (matches human pc_data.parquet)
pc["session_started_at"] = (
    pd.to_datetime(pc["session_started_at"], utc=True).astype("int64") / 1e9
)

# Melt PC1…PC6 into (pc_name, pc_value) rows — this is what makes it long format
pc_long = pc.melt(
    id_vars=[
        "generation_key", "vase",
        "is_selected_vase", "is_parent_vase",
        "participant_id", "session_id", "play_counter",
        "phase", "model", "gen",
        "wall_time_s", "rt_tokens", "budget_tokens", "budget_utilisation",
        "input_tokens", "output_tokens", "stop_reason",
        "session_started_at", "timestamp", "source_file",
    ],
    value_vars=PC_COLS,
    var_name="pc_name",
    value_name="pc_value",
)

# Match column order: shared structural cols first, then AI-specific extras
SHARED_COLS = [
    "generation_key", "vase", "pc_name", "pc_value",
    "is_selected_vase", "is_parent_vase",
    "participant_id", "session_id", "play_counter",
    "session_started_at",
]
AI_EXTRA_COLS = [
    "phase", "model", "gen",
    "wall_time_s", "rt_tokens", "budget_tokens", "budget_utilisation",
    "input_tokens", "output_tokens", "stop_reason",
    "timestamp", "source_file",
]
pc_long = pc_long[SHARED_COLS + AI_EXTRA_COLS]

print(f"PC rows : {len(pc_long):,}  ({len(pc_long) // len(PC_COLS):,} vases × 6 PCs)")
print(f"is_parent_vase dtype: {pc_long['is_parent_vase'].dtype}")
pc_long.head(6)

PC rows : 199,980  (33,330 vases × 6 PCs)
is_parent_vase dtype: bool


,generation_key,vase,pc_name,pc_value,is_selected_vase,is_parent_vase,participant_id,session_id,play_counter,session_started_at,...,gen,wall_time_s,rt_tokens,budget_tokens,budget_utilisation,input_tokens,output_tokens,stop_reason,timestamp,source_file
0,round_0_gen_1,vase_1,PC1,-3.374880,False,False,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,1,1.779840e+06,...,1.0,2.169575,69,5000,0.0164,1558,82,completed,2026-05-27T00:06:24.054391,gpt_results.csv.gz
1,round_0_gen_1,vase_2,PC1,-7.832178,True,False,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,1,1.779840e+06,...,1.0,2.169575,69,5000,0.0164,1558,82,completed,2026-05-27T00:06:24.054391,gpt_results.csv.gz
2,round_0_gen_2,vase_1,PC1,-6.031124,False,False,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,1,1.779840e+06,...,2.0,2.220651,122,5000,0.0270,1558,135,completed,2026-05-27T00:06:26.302597,gpt_results.csv.gz
3,round_0_gen_2,vase_2,PC1,-7.832178,True,True,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,1,1.779840e+06,...,2.0,2.220651,122,5000,0.0270,1558,135,completed,2026-05-27T00:06:26.302597,gpt_results.csv.gz
4,round_0_gen_3,vase_1,PC1,-9.644491,False,False,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,1,1.779840e+06,...,3.0,1.857240,98,5000,0.0222,1558,111,completed,2026-05-27T00:06:28.210096,gpt_results.csv.gz
5,round_0_gen_3,vase_2,PC1,-7.832178,True,True,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,5f62ef9d-9dd3-4518-9a1f-548aa38d2ac4,1,1.779840e+06,...,3.0,1.857240,98,5000,0.0222,1558,111,completed,2026-05-27T00:06:28.210096,gpt_results.csv.gz


In [ ]:
pc_out_path = f"{OUT_DIR}/ai_pc_data.parquet"
pc_long.to_parquet(pc_out_path, index=False)
print(f"Saved {len(pc_long):,} rows → {pc_out_path}")

# Schema comparison against human pc_data.parquet
human_pc = pd.read_parquet("../file_processing/processed/pc_data.parquet", 
                            filters=[('participant_id', 'in', 
                                      pd.read_parquet('../file_processing/processed/pc_data.parquet', 
                                                      columns=['participant_id'])['participant_id'].unique()[:2].tolist())])

shared    = sorted(set(human_pc.columns) & set(pc_long.columns))
mismatch  = [(c, str(human_pc[c].dtype), str(pc_long[c].dtype))
             for c in shared if str(human_pc[c].dtype) != str(pc_long[c].dtype)]

print(f"\nShared columns  : {len(shared)}")
print(f"Human-only cols : {sorted(set(human_pc.columns) - set(pc_long.columns))}")
print(f"AI-only cols    : {sorted(set(pc_long.columns) - set(human_pc.columns))}")
if mismatch:
    print(f"\nDtype mismatches:")
    for col, h, a in mismatch:
        print(f"  {col}: human={h}  ai={a}")
else:
    print("\nAll shared columns have matching dtypes")

Saved 199,980 rows → pca_llm_results/processed/ai_pc_data.parquet

Shared columns  : 10
Human-only cols : ['adjusted_decision_time_ms', 'adjusted_selection_elapsed_time_s', 'completion_status', 'decision_time_ms', 'metadata_schema_version', 'rt_calibration_median_ms', 'rt_calibration_status', 'rt_calibration_trials_n', 'rt_onset_correction_ms', 'selection_elapsed_time_s', 'session_duration_s']
AI-only cols    : ['budget_tokens', 'budget_utilisation', 'gen', 'input_tokens', 'model', 'output_tokens', 'phase', 'rt_tokens', 'source_file', 'stop_reason', 'timestamp', 'wall_time_s']

All shared columns have matching dtypes ✓
